In [2]:
source ../modules/setup.tcl

lassign "2018 18" year day

aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2
set input [string trim [aoc::get-input $year $day]]
jupyter::html "<h2>Input</h2>"
jupyter::display "text/plain" [string range $input 0 100]...;

(cached)


## --- Day 18: Settlers of The North Pole ---




On the outskirts of the North Pole base construction project, many Elves are collecting lumber.




The lumber collection area is 50 acres by 50 acres; each acre can be either <b>open ground</b> (`.`), <b>trees</b> (`|`), or a <b>lumberyard</b> (`#`). You take a scan of the area (your puzzle input).




Strange magic is at work here: each minute, the landscape looks entirely different. In exactly <b>one minute</b>, an open acre can fill with trees, a wooded acre can be converted to a lumberyard, or a lumberyard can be cleared to open ground (the lumber having been sent to other projects).




The change to each acre is based entirely on <b>the contents of that acre</b> as well as <b>the number of open, wooded, or lumberyard acres adjacent to it</b> at the start of each minute. Here, "adjacent" means any of the eight acres surrounding that acre. (Acres on the edges of the lumber collection area might have fewer than eight adjacent acres; the missing acres aren't counted.)




In particular:




- An <b>open</b> acre will become filled with <b>trees</b> if <b>three or more</b> adjacent acres contained trees. Otherwise, nothing happens.

- An acre filled with <b>trees</b> will become a <b>lumberyard</b> if <b>three or more</b> adjacent acres were lumberyards. Otherwise, nothing happens.

- An acre containing a <b>lumberyard</b> will remain a <b>lumberyard</b> if it was adjacent to <b>at least one other lumberyard and at least one acre containing trees</b>. Otherwise, it becomes <b>open</b>.




These changes happen across all acres <b>simultaneously</b>, each of them using the state of all acres at the beginning of the minute and changing to their new form by the end of that same minute. Changes that happen during the minute don't affect each other.




For example, suppose the lumber collection area is instead only 10 by 10 acres with this initial configuration:



```
Initial state:
.#.#...|#.
.....#|##|
.|..|...#.
..|#.....#
#.#|||#|#|
...#.||...
.|....|...
||...#|.#|
|.||||..|.
...#.|..|.

After 1 minute:
.......##.
......|###
.|..|...#.
..|#||...#
..##||.|#|
...#||||..
||...|||..
|||||.||.|
||||||||||
....||..|.

After 2 minutes:
.......#..
......|#..
.|.|||....
..##|||..#
..###|||#|
...#|||||.
|||||||||.
||||||||||
||||||||||
.|||||||||

After 3 minutes:
.......#..
....|||#..
.|.||||...
..###|||.#
...##|||#|
.||##|||||
||||||||||
||||||||||
||||||||||
||||||||||

After 4 minutes:
.....|.#..
...||||#..
.|.#||||..
..###||||#
...###||#|
|||##|||||
||||||||||
||||||||||
||||||||||
||||||||||

After 5 minutes:
....|||#..
...||||#..
.|.##||||.
..####|||#
.|.###||#|
|||###||||
||||||||||
||||||||||
||||||||||
||||||||||

After 6 minutes:
...||||#..
...||||#..
.|.###|||.
..#.##|||#
|||#.##|#|
|||###||||
||||#|||||
||||||||||
||||||||||
||||||||||

After 7 minutes:
...||||#..
..||#|##..
.|.####||.
||#..##||#
||##.##|#|
|||####|||
|||###||||
||||||||||
||||||||||
||||||||||

After 8 minutes:
..||||##..
..|#####..
|||#####|.
||#...##|#
||##..###|
||##.###||
|||####|||
||||#|||||
||||||||||
||||||||||

After 9 minutes:
..||###...
.||#####..
||##...##.
||#....###
|##....##|
||##..###|
||######||
|||###||||
||||||||||
||||||||||

After 10 minutes:
.||##.....
||###.....
||##......
|##.....##
|##.....##
|##....##|
||##.####|
||#####|||
||||#|||||
||||||||||

```



After 10 minutes, there are `37` wooded acres and `31` lumberyards.  Multiplying the number of wooded acres by the number of lumberyards gives the total <b>resource value</b> after ten minutes: `37 * 31 = <b>1147</b>`.




<b>What will the total resource value of the lumber collection area be after 10 minutes?</b>




(cached)



## --- Part Two ---



This important natural resource will need to last for at least thousands of years.  Are the Elves collecting this lumber sustainably?




<b>What will the total resource value of the lumber collection area be after 1000000000 minutes?</b>




(cached)

Input

||.|..#|.|.....|..|........|..#..|#...|.........#.
...#....|#..|......#.|||...#...|...#....|.......#....

In [10]:
proc step state {
    set sx [llength [lindex $state 0]]
    set sy [llength $state]
    set new $state
    for {set x 0} {$x < $sx} {incr x} {
         for {set y 0} {$y < $sy} {incr y} {
            set surroundings {}
            foreach nb [aoc::neighbours8 $x $y] {
                lassign $nb nx ny
                if {$nx < 0 || $nx > $sx || $ny < 0 || $ny > $sy} continue
                append surroundings [lindex $state $ny $nx]
            }
            set curr [lindex $state $y $x]
            lset new $y $x $curr
            switch -exact $curr {
                . {
                    if {[aoc::count $surroundings |] >= 3} {lset new $y $x |}
                }
                | {
                    if {[aoc::count $surroundings #] >= 3} {lset new $y $x #}
                }
                # {
                    if {[aoc::count $surroundings #] == 0 || [aoc::count $surroundings |] == 0} {lset new $y $x .}
                }
            }
        }
    }
    return $new

}

proc aoc::part1 input {
    set state [lmap l [split $input \n] {split $l ""}]
#     display $state
#     puts \n=======\n
    time {set state [step $state]} 10
    # display $state
    * [aoc::count $state |] [aoc::count $state #] 
    }

proc aoc::part2 input {
    set state [lmap l [split $input \n] {split $l ""}]
    # display $state
    # puts \n=======\n
    set time 0
    while {1} {
        if {[info exists seen($state)]} {
            set startcycle $seen($state)
            set endcycle $time
            set period [- $endcycle $startcycle]
            break
        }
        set seen($state) $time
        set seen($time) [* [aoc::count $state |] [aoc::count $state #]]
        set state [step $state]
        incr time
    } 

    return  $seen([+ $startcycle [% [- 1000000000 $startcycle] $period]])
}
aoc::results 

Part1	486878 (182587 us)
Part2	190836 (7877698 us)


486878 190836

In [12]:
set test {.#.#...|#.
.....#|##|
.|..|...#.
..|#.....#
#.#|||#|#|
...#.||...
.|....|...
||...#|.#|
|.||||..|.
...#.|..|.}
aoc::showgrid [aoc::togrid $test];


10x10
.#.#...|#.
.....#|##|
.|..|...#.
..|#.....#
#.#|||#|#|
...#.||...
.|....|...
||...#|.#|
|.||||..|.
...#.|..|.


In [3]:
aoc::togrid {@@##111}

{@ @ # # 1 1 1}

In [34]:
proc htmlgrid {grid {boxsize 10} {colors {}}} {
    set width [llength [lindex $grid 0]]
    set height [llength $grid]
    set html "<p>"
    for {set y 0} {$y < $height} {incr y} {
        for {set x 0} {$x < $width} {incr x} {
            lappend html "<p style\"color=\"#123456\">[lindex $grid $y $x]</color>"
        }
    }
    puts [join $html ""]
    jupyter::display "text/html" [join $html ""]
}
htmlgrid {! @ # ! # # #}

<p><p style"color="#123456">!</color><p style"color="#123456">@</color><p style"color="#123456">#</color><p style"color="#123456">!</color><p style"color="#123456">#</color><p style"color="#123456">#</color><p style"color="#123456">#</color>


! @ # ! # # #

display-id-33

display-id-8